<a href="https://colab.research.google.com/github/adrianamimo/dark-matter-ngc3198-simulation/blob/main/Stability_darkmatter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

n = 1000
G = 6.67e-11
M = 6.6821e40
r_min = 3.086e19
r_max = 1.1e21
t = 1.5768e14
t_sub = t / 20
veces = 4000
M_oscura = 3.04479e41
a = 9.7384e20
p_0 = 1.049e-22
h_R = 8.2696e19

plt.rcParams["animation.embed_limit"] = 200

np.random.seed(60)

angulos = np.random.uniform(0, 360, n)
angulos = np.radians(angulos)

b = r_max / 5
radios = np.random.exponential(scale=b, size=n)
radios = np.clip(radios, r_min, r_max)
x = radios * np.cos(angulos)
y = radios * np.sin(angulos)
posiciones = np.column_stack((x, y))

M_encerrada_visible = M * (1 - np.exp(-radios / h_R) * (1 + radios / h_R))
v = np.sqrt((G * M_encerrada_visible)/ radios)

M_oscura_encerrada = 4 * np.pi * p_0 * (a**3) * (np.log(1 + radios / a) - radios / (a + radios))
M_encerrada = M_encerrada_visible + M_oscura_encerrada
v_oscura = np.sqrt((G * M_oscura_encerrada)/ radios)
v_total = np.sqrt(v**2 + v_oscura**2)

vx = -v_total * np.sin(angulos)
vy = v_total * np.cos(angulos)
velocidades = np.column_stack((vx, vy))

def datos(fotograma):
  global posiciones, velocidades
  for _ in range(20):
    radios = np.sqrt(posiciones[:, 0]**2 + posiciones[:, 1]**2)

    M_encerrada_visible_actual = M * (1 - np.exp(-radios / h_R) * (1 + radios / h_R))
    if M_oscura != 0:
      M_oscura_encerrada2 = 4 * np.pi * p_0 * (a**3) * (np.log(1 + radios / a) - radios / (a + radios))
      M_encerrada2= M_encerrada_visible_actual + M_oscura_encerrada2
    else:
      M_encerrada2 = M_encerrada_visible_actual

    g = (G * M_encerrada2) / (radios**2)
    ax = -g * (posiciones[:, 0] / radios)
    ay = -g * (posiciones[:, 1] / radios)
    aceleraciones = np.column_stack((ax, ay))

    f_gravitatoria = (G * M_encerrada) / (radios**2)
    f_centripeta = (velocidades[:, 0]**2 + velocidades[:, 1]**2) / radios

    velocidades += aceleraciones * t_sub
    posiciones += velocidades * t_sub

    dibujo.set_offsets(posiciones)
    return dibujo,

fig, ax = plt.subplots()
ax.set_xlim(-r_max * 2, r_max * 2)
ax.set_ylim(-r_max * 2, r_max * 2)
ax.set_facecolor("k")
fig.set_facecolor("k")
ax.set_title("Simulación de Galaxia con Materia Oscura", color="white")

dibujo = ax.scatter(posiciones[:, 0], posiciones[:, 1], s=1, c="white")

animacion = animation.FuncAnimation(fig, datos, frames=veces, interval=10, cache_frame_data=False)
animacion_html = HTML(animacion.to_jshtml())
plt.close(fig)
display(animacion_html)